# Geo-Wiki 2015: acquire the reference data

Purpose: download and preserve the complete Geo-Wiki reference CSV and the
country boundaries needed by the [exploration notebook](0.03-tp-geowiki-2015-exploration.ipynb).
Inputs: [Zenodo record 14871660, v1](https://zenodo.org/records/14871660)
(CC BY 4.0) and Natural Earth 1:110m countries (public domain).
The reference year is **2015**, rather than the annotation timestamp or publication year.

Outputs: unchanged `data/raw/geowiki/14871660/final_reference_data.csv`, the
record metadata beside it, and `data/external/natural_earth/countries.geojson`.
Source notes are in the README files beside these datasets. The CSV is approximately
2 GB; this notebook does not filter annotations, infer raster grids, or export TIFFs.
It separates acquisition from the preparation and QGIS exports in the supplied
`0.03-tp-geowiki-2015-sites.ipynb`.


```mermaid
flowchart TD
    A["Zenodo 14871660: pinned CSV and metadata"] --> B["Download or reuse existing full file"]
    B --> C["Verify published size and MD5"]
    C --> D["data/raw/geowiki/14871660/"]
    E["Natural Earth 1:110m countries; pinned SHA256"] --> F["Download and verify unchanged GeoJSON"]
    F --> G["data/external/natural_earth/countries.geojson"]
    D --> H["Inspect CSV header; report acquisition status"]
    G --> H
```

This diagram describes the workflow. Mermaid rendering depends on the notebook viewer.


## 1. Environment and settings

Launch with `pixi run --locked notebook` and use its Python kernel. Paths come from
the installed thesis package and work from this nested notebook directory.
Existing complete files are verified and preserved; incomplete downloads use a
separate `.part` file and are never accepted as the original CSV.
Cached metadata allows offline verification after acquisition. Delete only the cached
metadata if you deliberately want to retrieve it again from the pinned record.

| Parameter | Default | Unit | Meaning / why this default |
| --- | --- | --- | --- |
| RECORD_ID | 14871660 | record ID | Pin the reference release used by the supplied notebooks. |
| FILE_NAME | final_reference_data.csv | n/a | Acquire the complete original annotation table. |
| EXPECTED_MD5 | fa3c8699d479d30bdd15d7ee229da563 | n/a | Published checksum for this release; reject a changed source. |
| BOUNDARY_SHA256 | 6866c877…c977f | n/a | Pin the exact boundary bytes used by the supplied workflow. |

The boundary URL follows Natural Earth's GitHub mirror, so its SHA256 pins the
content even if the upstream branch changes. A mismatch requires a deliberate
source/version review; it does not silently change the Africa definition.


In [1]:
from datetime import datetime, timezone
import hashlib
import json
import sys

import pandas as pd
import requests
from IPython.display import display

from tomer_peled_mgi_thesis.config import RAW_DATA_DIR, EXTERNAL_DATA_DIR, ensure_output_dir

RECORD_ID = 14871660
FILE_NAME = "final_reference_data.csv"
EXPECTED_MD5 = "fa3c8699d479d30bdd15d7ee229da563"
BOUNDARY_URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
BOUNDARY_SHA256 = "6866c877d39cba9c357620878839b336d569f8c662d3cfab4cb1dbe2d39c977f"

raw_dir = ensure_output_dir(RAW_DATA_DIR / "geowiki" / str(RECORD_ID))
boundary_dir = ensure_output_dir(EXTERNAL_DATA_DIR / "natural_earth")
source_path = raw_dir / FILE_NAME
metadata_path = raw_dir / "zenodo_record.json"
boundary_path = boundary_dir / "countries.geojson"
print(f"Python: {sys.executable}")
print(f"CSV: {source_path}")


Python: C:\Users\tomer\OneDrive\Home\MGI\Thesis\tomer_peled_mgi_thesis\.pixi\envs\default\python.exe
CSV: C:\Users\tomer\OneDrive\Home\MGI\Thesis\tomer_peled_mgi_thesis\data\raw\geowiki\14871660\final_reference_data.csv


## 2. Read the pinned source metadata

Retrieve the public record once and keep its original metadata. Confirm that its
file checksum matches the pinned release before using its download link. These
are integrity checks at the acquisition boundary, rather than repeated analytical audits.


In [2]:
record_url = f"https://zenodo.org/api/records/{RECORD_ID}"
if metadata_path.exists():
    record = json.loads(metadata_path.read_text(encoding="utf-8-sig"))
    print("Using cached Zenodo metadata")
else:
    response = requests.get(record_url, timeout=(30, 120))
    response.raise_for_status()
    record = response.json()
    metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")

file_info = next(item for item in record["files"] if item["key"] == FILE_NAME)
if file_info["checksum"] != f"md5:{EXPECTED_MD5}":
    raise ValueError("Zenodo metadata differs from the pinned CSV release")
download_url = file_info["links"]["self"]
print(f"Source: {record['metadata']['title']}")
print(f"License: {record['metadata']['license']['id']}; expected bytes: {file_info['size']:,}")


Using cached Zenodo metadata
Source: Global land cover data set at 10m for 2015 (Geo-Wiki)
License: cc-by-4.0; expected bytes: 1,974,819,745


## 3. Download or verify the complete CSV

Stream the large download to disk and print progress every 100 MiB. Verify both
file length and the publisher's MD5 before publishing a new file. Existing originals
are never overwritten, including when verification fails. An interrupted `.part`
download is restarted on the next run; partial HTTP previews are not analysis inputs.


In [3]:
downloaded = not source_path.exists()
candidate = source_path.with_suffix(".csv.part") if downloaded else source_path
if downloaded:
    received = 0
    with requests.get(download_url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with candidate.open("wb") as handle:
            for block in response.iter_content(chunk_size=1024 * 1024):
                handle.write(block)
                received += len(block)
                if received // (100 * 1024**2) != (received - len(block)) // (100 * 1024**2):
                    print(f"Downloaded {received / file_info['size']:.1%}", flush=True)

if candidate.stat().st_size != file_info["size"]:
    raise ValueError("CSV length differs from the published full file; do not use this input")
with candidate.open("rb") as handle:
    actual_md5 = hashlib.file_digest(handle, "md5").hexdigest()
if actual_md5 != EXPECTED_MD5:
    raise ValueError("CSV checksum differs from the published file; original was not overwritten")
if downloaded:
    candidate.rename(source_path)

print(f"{'Downloaded' if downloaded else 'Reused'} complete CSV: {source_path.stat().st_size:,} bytes")
print(f"Verified MD5: {actual_md5}")
print(f"Verified at UTC: {datetime.now(timezone.utc).isoformat()}")


Reused complete CSV: 1,974,819,745 bytes
Verified MD5: fa3c8699d479d30bdd15d7ee229da563
Verified at UTC: 2026-10-14T15:19:19.108312+00:00


## 4. Acquire the Africa selection reference

Keep the original Natural Earth country GeoJSON unchanged in `data/external/`.
This is the same coarse 1:110m boundary content as the supplied notebook.
[Natural Earth data are public domain](https://www.naturalearthdata.com/about/terms-of-use/).
It is suitable for a continent overview, but can omit small islands and coastal detail.


In [4]:
boundary_downloaded = not boundary_path.exists()
if boundary_downloaded:
    response = requests.get(BOUNDARY_URL, timeout=(30, 120))
    response.raise_for_status()
    boundary_bytes = response.content
else:
    boundary_bytes = boundary_path.read_bytes()
if hashlib.sha256(boundary_bytes).hexdigest() != BOUNDARY_SHA256:
    raise ValueError("Natural Earth boundary changed; review the source before changing its pinned checksum")
if boundary_downloaded:
    boundary_path.write_bytes(boundary_bytes)
print(f"{'Downloaded' if boundary_downloaded else 'Reused'} verified boundary: {boundary_path}")


Downloaded verified boundary: C:\Users\tomer\OneDrive\Home\MGI\Thesis\tomer_peled_mgi_thesis\data\external\natural_earth\countries.geojson


## 5. Inspect the handoff

Read only five rows to make the source columns visible. Zenodo documents
`sampleid` as a cluster identifier, `submission_item_id` as an annotation pixel
identifier, and `center_x`/`center_y` as WGS84 longitude/latitude in degrees.
The unlabeled first CSV column is an export index, not a site identifier.
The exploration notebook reads the complete file in chunks.


In [5]:
preview = pd.read_csv(source_path, nrows=5)
display(preview.drop(columns=[column for column in preview if column.startswith("Unnamed:")]))
display(pd.DataFrame([
    {"input": "Geo-Wiki CSV", "path": str(source_path), "status": "complete; size and MD5 verified"},
    {"input": "Natural Earth countries", "path": str(boundary_path), "status": "SHA256 verified"},
]))


,validation_id,submission_item_id,sampleid,timestamp,class_id,class_name,reference_year,center_x,center_y
0,907283,15220761,998380,2017-11-25 17:27:19.545618,3027,crops,2015,76.875177,27.500164
1,907283,15220766,998380,2017-11-25 17:27:19.545618,3024,tree,2015,76.875378,27.500071
2,907506,15242996,1000543,2017-11-26 15:56:21.282651,3026,grassland,2015,44.913569,-25.095318
3,907724,15264870,1010739,2017-11-28 12:17:54.465365,3026,grassland,2015,18.749929,13.124710
4,907724,15264900,1010739,2017-11-28 12:17:54.465365,3026,grassland,2015,18.749930,13.124619


,input,path,status
0,Geo-Wiki CSV,C:\Users\tomer\OneDrive\Home\MGI\Thesis\tomer_...,complete; size and MD5 verified
1,Natural Earth countries,C:\Users\tomer\OneDrive\Home\MGI\Thesis\tomer_...,SHA256 verified


## 6. Conclusions and reproduction

A successful run establishes complete, byte-verified inputs for exploration.
Checksums establish file integrity; they do not establish annotation accuracy.
No Africa or class filtering happens here. The reference year is 2015 and the
source coordinates remain EPSG:4326; no CRS transformation is performed.

Next, restart and run all of `0.03-tp-geowiki-2015-exploration.ipynb`.
From the project root, reproduce this notebook with:
`pixi run --locked check-geowiki-download`.
Executed copies go to `runs/notebooks/`. First acquisition needs internet access;
subsequent runs can verify the cached metadata and files offline. Source notebooks
and environment changes remain uncommitted, so keep them with the lockfile for reproduction.
